# Gold Layer: UC Metric Views (Parker Aerospace)

Standardized KPIs for the Parker installed-base platform, in `genie_zeroops_mfg_catalog.default`:
- **component_health_metrics** (`gold_component_health`): health index, failure risk tier, warranty exposure, compliance
- **part_failure_metrics** (`ml_component_failure_forecast`): next-quarter forecast KPIs from the leakage-free point-in-time model (MLflow run `6b20bf8b32c84774807ed160317fa799`). This view previously read `ml_part_failure_predictions`, which was retired for target leakage (0.999 AUC).
- **maintenance_anomaly_metrics** (`gold_maintenance_anomalies`): anomaly frequency, severity, detection latency, warranty impact, MRO quality

All three cells use `ALTER VIEW`, so re-running them keeps existing grants. The YAML in the component-health and anomaly cells is the live definition as of 2026-11-02, so re-running those is a no-op. On a fresh workspace where the views don't exist yet, swap the first line for the `CREATE VIEW ... WITH METRICS LANGUAGE YAML AS $$` line given in each cell's header comment. The last cell validates all three views.

In [0]:
-- ALTER VIEW keeps the existing grants. The YAML below is the live definition (2026-11-03), so re-running is a no-op.
-- Fresh workspace (view does not exist yet): replace the first line with
--   CREATE VIEW genie_zeroops_mfg_catalog.default.component_health_metrics WITH METRICS LANGUAGE YAML AS $$
ALTER VIEW genie_zeroops_mfg_catalog.default.component_health_metrics AS $$
version: 1.1
source: genie_zeroops_mfg_catalog.default.gold_component_health
comment: "Parker component health KPIs: fleet-wide health index, failure risk distribution, warranty exposure, and maintenance compliance by airline and component type."
dimensions:
  - name: airline
    expr: airline
    comment: Airline customer operating the component
    synonyms:
      - carrier
      - customer
  - name: component_type
    expr: component_type
    comment: Parker component category
    synonyms:
      - part type
      - component category
  - name: criticality
    expr: criticality
    comment: FLIGHT_CRITICAL or ESSENTIAL
  - name: failure_risk_category
    expr: failure_risk_category
    comment: Rule-based risk tier from health index and overdue service (LOW/MEDIUM/HIGH/CRITICAL), not a model output
    synonyms:
      - risk level
      - risk tier
  - name: parker_division
    expr: parker_division
    comment: Parker Hannifin business division
  - name: aircraft_type
    expr: aircraft_type
    comment: Aircraft model designation
measures:
  - name: Total Components
    expr: COUNT(1)
    comment: Number of installed Parker components
    display_name: Total Components
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Avg Health Index
    expr: AVG(health_index)
    comment: Average component health score, 0-100, higher is healthier
    display_name: Avg Health Index
    format:
      type: number
      decimal_places:
        type: exact
        places: 1
  - name: Critical Risk Components
    expr: COUNT(1) FILTER (WHERE failure_risk_category = 'CRITICAL')
    comment: Components in CRITICAL failure risk tier
    display_name: Critical Risk Components
  - name: High Risk Components
    expr: "COUNT(1) FILTER (WHERE failure_risk_category IN ('CRITICAL', 'HIGH'))"
    comment: Components needing priority maintenance
    display_name: High + Critical Components
  - name: Active Anomalies
    expr: COUNT(1) FILTER (WHERE has_active_anomaly = true)
    comment: Components with unresolved maintenance anomalies
    display_name: Active Anomalies
  - name: Total Warranty Exposure
    expr: SUM(estimated_warranty_exposure_usd)
    comment: Aggregate warranty liability across all components
    display_name: Total Warranty Exposure ($)
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
  - name: Avg Days to Failure
    expr: AVG(days_to_predicted_failure)
    comment: Simulated days-to-failure estimate, banded by health index, not a model prediction
    display_name: Avg Est. Days to Failure (simulated)
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Avg Compliance Rate
    expr: AVG(maintenance_compliance_rate)
    comment: Average compliance rate across components
    display_name: Avg Maintenance Compliance Rate
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1
  - name: Total Anomaly Events
    expr: SUM(anomaly_event_count)
    comment: Sum of anomalous maintenance events
    display_name: Total Anomaly Events
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
$$

In [0]:
-- Repointed from the retired ml_part_failure_predictions (target leakage) to the leakage-free forecast.
-- Use ALTER VIEW so existing grants are kept; on a fresh workspace use CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML.
ALTER VIEW genie_zeroops_mfg_catalog.default.part_failure_metrics AS $$
version: 1.1
source: genie_zeroops_mfg_catalog.default.ml_component_failure_forecast
comment: "Next-quarter component failure forecast KPIs from the leakage-free point-in-time model (MLflow run 6b20bf8b32c84774807ed160317fa799). Replaces the retired ml_part_failure_predictions model."
dimensions:
  - expr: airline
  - expr: component_type
  - expr: criticality
  - expr: recommended_action
  - expr: risk_decile
    comment: Forecast risk decile, 10 = highest-risk 10% of components
measures:
  - name: Total Components
    expr: COUNT(1)
  - name: Avg Forecast Probability
    expr: AVG(forecast_probability)
    format: {type: percentage, decimal_places: {type: exact, places: 1}}
  - name: Top-Decile Components
    expr: COUNT(1) FILTER (WHERE risk_decile = 10)
  - name: Proactive-Visit Components
    expr: COUNT(1) FILTER (WHERE recommended_action = 'PROACTIVE_FIELD_SERVICE_VISIT')
  - name: Expected Warranty Exposure
    expr: SUM(expected_warranty_exposure_usd)
    format: {type: currency, currency_code: USD, abbreviation: compact}
  - name: Next-Quarter Events (back-test)
    expr: SUM(actual_next_quarter_event)
    comment: Components that actually had a HIGH/CRITICAL or warranty-risk event in the forecast quarter
$$

In [0]:
-- ALTER VIEW keeps the existing grants. The YAML below is the live definition (2026-11-02), so re-running is a no-op.
-- Fresh workspace (view does not exist yet): replace the first line with
--   CREATE VIEW genie_zeroops_mfg_catalog.default.maintenance_anomaly_metrics WITH METRICS LANGUAGE YAML AS $$
ALTER VIEW genie_zeroops_mfg_catalog.default.maintenance_anomaly_metrics AS $$
version: 1.1
source: genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
comment: "Maintenance anomaly analysis KPIs: anomaly frequency by type, severity distribution, detection latency, warranty impact, and MRO quality tracking."
dimensions:
  - name: airline
    expr: airline
    comment: Airline where anomaly was detected
  - name: component_type
    expr: component_type
    comment: Parker component category
  - name: anomaly_type
    expr: anomaly_type
    comment: Anomaly classification (6 types)
    synonyms:
      - anomaly category
      - defect type
  - name: anomaly_severity
    expr: anomaly_severity
    comment: Severity tier (LOW/MEDIUM/HIGH/CRITICAL)
    synonyms:
      - severity
      - severity level
  - name: flight_delay_risk
    expr: flight_delay_risk
    comment: Associated flight delay risk level
  - name: mro_shop
    expr: mro_shop
    comment: MRO facility where anomaly was found
  - name: criticality
    expr: criticality
    comment: Component criticality classification
  - name: parker_division
    expr: parker_division
    comment: Parker division responsible for component
measures:
  - name: Total Anomalies
    expr: COUNT(1)
    comment: Total anomaly events detected
    display_name: Total Anomalies
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Critical Anomalies
    expr: COUNT(1) FILTER (WHERE anomaly_severity = 'CRITICAL')
    comment: CRITICAL severity anomalies
    display_name: Critical Anomalies
  - name: High Delay Risk
    expr: "COUNT(1) FILTER (WHERE flight_delay_risk IN ('CRITICAL', 'HIGH'))"
    comment: Anomalies with HIGH or CRITICAL flight delay risk
    display_name: High Delay Risk Events
  - name: Avg Days Undetected
    expr: AVG(days_undetected)
    comment: Average days anomaly went undetected
    display_name: Avg Days Undetected
    format:
      type: number
      decimal_places:
        type: exact
        places: 1
  - name: Warranty Risk Events
    expr: COUNT(1) FILTER (WHERE warranty_claim_risk = true)
    comment: Anomalies that may trigger warranty claims
    display_name: Warranty Risk Events
  - name: Total Warranty Exposure
    expr: SUM(record_warranty_exposure_usd)
    comment: Aggregate warranty exposure from anomaly events
    display_name: Total Warranty Exposure ($)
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
  - name: Overdue Events
    expr: COUNT(1) FILTER (WHERE is_overdue = true)
    comment: Anomalies associated with overdue maintenance
    display_name: Overdue Maintenance Events
  - name: Avg Overdue Days
    expr: AVG(overdue_days) FILTER (WHERE overdue_days > 0)
    comment: Average days overdue for overdue events
    display_name: Avg Overdue Days
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
$$

In [0]:
SELECT 'component_health_metrics' AS metric_view, MEASURE(`Total Components`) AS value, 'components' AS unit FROM genie_zeroops_mfg_catalog.default.component_health_metrics
UNION ALL
SELECT 'part_failure_metrics', MEASURE(`Proactive-Visit Components`), 'proactive-visit components' FROM genie_zeroops_mfg_catalog.default.part_failure_metrics
UNION ALL
SELECT 'part_failure_metrics', MEASURE(`Top-Decile Components`), 'top-decile components' FROM genie_zeroops_mfg_catalog.default.part_failure_metrics
UNION ALL
SELECT 'maintenance_anomaly_metrics', MEASURE(`Total Anomalies`), 'anomalies' FROM genie_zeroops_mfg_catalog.default.maintenance_anomaly_metrics